# General VLMs on FineSightBench (HF Hub)

Evaluate all tested VLMs **except** `deepseek-vl2-tiny` on the `Volavion/FineSightBench` dataset.
These models all load via standard `transformers` / `AutoProcessor` — no monkey-patching needed.

**Pipeline**
1. Load dataset from HF Hub.
2. Sample **2 items per task type** from each split (perception / reasoning).
3. For each model: load → run inference → JSON eval → unload.
4. Aggregate strict accuracy, mean score, hallucination rate per model and save reports.


In [ ]:
# ── Cell 1: Imports & Setup ───────────────────────────────────────────────────
import os, sys, json, re as _re, time, warnings
from pathlib import Path
from collections import defaultdict
import random

warnings.filterwarnings("ignore")

import numpy as np
import torch
from PIL import Image
import matplotlib
matplotlib.use("Agg")
from IPython.display import display
import pandas as pd

from datasets import load_dataset

# Force local repo path so editable install is always preferred
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "finesightbench").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT))
for _m in [m for m in list(sys.modules) if m == "finesightbench" or m.startswith("finesightbench.")]:
    sys.modules.pop(_m, None)

from finesightbench.evaluation import (
    FieldSpec, ORDERED_LIST, MAPPING,
    evaluate_json_prediction, aggregate_json_results,
)

# ── Constants ─────────────────────────────────────────────────────────────────
HF_DATASET_ID = "Volavion/FineSightBench"
MAX_PER_TASK  = 2
SEED          = 42
OUTPUT_DIR    = REPO_ROOT / "outputs" / "vlm_eval_hf_general"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Output : {OUTPUT_DIR}")


In [ ]:
# ── Cell 2: Load FineSightBench from HF Hub ───────────────────────────────────
print(f"Loading dataset: {HF_DATASET_ID}")
ds = load_dataset(HF_DATASET_ID)
print(ds)

for split_name in ds.keys():
    s0 = ds[split_name][0]
    print(f"\n{split_name} sample 0:")
    print(f"  image_id  : {s0['image_id']}")
    print(f"  task_type : {s0['task_type']}")
    print(f"  difficulty: {s0['difficulty']}")
    print(f"  question  : {s0['question'][:80]}...")
    print(f"  answer    : {s0['answer']}")


In [ ]:
# ── Cell 3: Model Registry (all non-DeepSeek tested VLMs) ────────────────────
# All entries use the same HuggingFaceVLM adapter from the framework, which
# calls AutoProcessor + AutoModelForCausalLM/ImageTextToText with eager attn.
from finesightbench.evaluation.framework import (
    MODEL_SPECS, HuggingFaceVLM, list_supported_models,
)

EXCLUDE = {"deepseek-vl2-tiny"}
MODEL_NAMES = [m for m in list_supported_models() if m not in EXCLUDE]

print("Models selected for this run:")
for name in MODEL_NAMES:
    spec = MODEL_SPECS[name]
    note = f"  [{spec.notes}]" if spec.notes else ""
    print(f"  {name:40s}  {spec.model_id}{note}")


In [ ]:
# ── Cell 4: Build Subset — 2 samples per task type per split ─────────────────
rng = random.Random(SEED)

_by_task: dict[str, list[tuple[str, dict]]] = defaultdict(list)
for split_name in ("perception", "reasoning"):
    for item in ds[split_name]:
        _by_task[str(item["task_type"])].append((split_name, item))

subset: dict[str, list[dict]] = {"perception": [], "reasoning": []}
for task, items in _by_task.items():
    rng.shuffle(items)
    chosen = items[:MAX_PER_TASK]
    for split_name, item in chosen:
        subset[split_name].append(item)

print("Samples per task_type (up to 2 each):")
for task in sorted(_by_task):
    n = sum(1 for s in subset["perception"] + subset["reasoning"]
            if s["task_type"] == task)
    print(f"  {task:30s} {n}")
print(f"\nTotal: perception={len(subset['perception'])}, reasoning={len(subset['reasoning'])}")


In [ ]:
# ── Cell 5: JSON Evaluation Schemas & Gold-Answer Parser ─────────────────────
# Identical to the DeepSeek notebook so results are directly comparable.

TASK_SCHEMAS = {
    "letter_recognition":      [FieldSpec("letter")],
    "animal_recognition":      [FieldSpec("animal")],
    "block_recognition":       [FieldSpec("present")],
    "color_block_recognition": [FieldSpec("color")],
    "shape_recognition":       [FieldSpec("shape")],
    "chain_reasoning":         [FieldSpec("objects", kind=ORDERED_LIST)],
    "comparison_chain":        [FieldSpec("objects", kind=ORDERED_LIST)],
    "counting_chain":          [FieldSpec("counts", kind=MAPPING), FieldSpec("total")],
    "blur_chain":              [FieldSpec("counts", kind=MAPPING), FieldSpec("total")],
}

# Per-task JSON-format instruction appended to the question
TASK_JSON_INSTR = {
    "letter_recognition":
        'Respond ONLY with JSON like {"letter": "A"}.',
    "animal_recognition":
        'Respond ONLY with JSON like {"animal": "cat"}.',
    "block_recognition":
        'Respond ONLY with JSON like {"present": "yes"} or {"present": "no"}.',
    "color_block_recognition":
        'Respond ONLY with JSON like {"color": "red"}.',
    "shape_recognition":
        'Respond ONLY with JSON like {"shape": "circle"}.',
    "chain_reasoning":
        'Respond ONLY with JSON like {"objects": ["<color> dot", "<color> dot", ...]} '
        'listing the dots from top to bottom.',
    "comparison_chain":
        'Respond ONLY with JSON like {"objects": ["<color> <animal>", ...]} '
        'from smallest to largest.',
    "counting_chain":
        'Respond ONLY with JSON like {"counts": {"<color>": <int>, ...}, "total": <int>}.',
    "blur_chain":
        'Respond ONLY with JSON like {"counts": {"<shape>": <int>, ...}, "total": <int>}.',
}


def _parse_counts_total(ans: str) -> dict:
    """'1 purple, 1 green; total: 2' -> {'counts': {'purple':1,'green':1}, 'total':2}."""
    counts_part, _, total_part = ans.partition(";")
    counts: dict[str, int] = {}
    for chunk in counts_part.split(","):
        chunk = chunk.strip()
        if not chunk:
            continue
        m = _re.match(r"(\d+)\s+(.+)", chunk)
        if m:
            counts[m.group(2).strip().lower()] = int(m.group(1))
    tm = _re.search(r"total\s*:\s*(\d+)", total_part)
    total = int(tm.group(1)) if tm else sum(counts.values())
    return {"counts": counts, "total": total}


def build_gt_obj(task_type: str, gold: str) -> dict:
    g = gold.strip()
    if task_type == "letter_recognition":
        return {"letter": g}
    if task_type == "animal_recognition":
        return {"animal": g.lower()}
    if task_type == "block_recognition":
        return {"present": g.lower()}
    if task_type == "color_block_recognition":
        return {"color": g.lower()}
    if task_type == "shape_recognition":
        return {"shape": g.lower()}
    if task_type in ("chain_reasoning", "comparison_chain"):
        items = [p.strip().lower() for p in g.split(",") if p.strip()]
        return {"objects": items}
    if task_type in ("counting_chain", "blur_chain"):
        return _parse_counts_total(g.lower())
    return {"answer": g}


# Sanity check
_seen = {}
for split in ("perception", "reasoning"):
    for s in subset[split]:
        _seen.setdefault(s["task_type"], s)
for t, s in _seen.items():
    print(f"{t:30s} gold={s['answer']!r:30s} -> {build_gt_obj(t, s['answer'])}")


In [ ]:
# ── Cell 6: Benchmark Loop — one model at a time ─────────────────────────────
# Each model is loaded, evaluated on the shared subset, then unloaded to free
# GPU memory before the next model starts.

all_rows: list[dict] = []
model_elapsed: dict[str, float] = {}

for model_name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    adapter = HuggingFaceVLM(
        MODEL_SPECS[model_name],
        local_files_only=False,   # allow downloading from HF Hub
    )
    try:
        adapter.load()
    except Exception as exc:
        print(f"  [LOAD FAILED] {exc}")
        model_elapsed[model_name] = 0.0
        continue

    t0 = time.time()
    model_rows: list[dict] = []

    for split in ("perception", "reasoning"):
        for i, s in enumerate(subset[split]):
            img = s["image"]
            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.asarray(img))
            img = img.convert("RGB")

            task   = s["task_type"]
            instr  = TASK_JSON_INSTR.get(task, "Respond ONLY with a short JSON answer.")
            prompt = f"{s['question']}\n\n{instr}"

            try:
                raw_pred   = adapter.predict(img, prompt)
                infer_error = ""
            except Exception as e:
                raw_pred   = ""
                infer_error = f"{type(e).__name__}: {e}"

            schema = TASK_SCHEMAS.get(task, [FieldSpec("answer")])
            gt_obj = build_gt_obj(task, s["answer"])
            result = evaluate_json_prediction(raw_pred, gt_obj, schema)

            row = {
                "model_name":   model_name,
                "split":        split,
                "task_type":    task,
                "image_id":     s["image_id"],
                "gold_text":    s["answer"],
                "gold_obj":     gt_obj,
                "raw_pred":     raw_pred,
                "infer_error":  infer_error,
                "json_eval":    result.to_dict(),
            }
            model_rows.append(row)

            status = "OK" if not infer_error else "ERR"
            print(
                f"  [{split}] {i+1}/{len(subset[split])} task={task:28s} "
                f"parse={result.parse_ok} score={result.overall_score:.2f} "
                f"strict={result.all_fields_matched} [{status}]"
            )

    elapsed = time.time() - t0
    model_elapsed[model_name] = elapsed
    print(f"  Elapsed: {elapsed:.1f}s  rows: {len(model_rows)}")

    all_rows.extend(model_rows)

    adapter.unload()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nTotal rows collected: {len(all_rows)}")


In [ ]:
# ── Cell 7: Aggregate & Report Results ───────────────────────────────────────
summary = aggregate_json_results(all_rows, group_keys=("model_name", "split", "task_type"))

print("=== Overall (all models combined) ===")
print(f"  n                  : {summary['n']}")
print(f"  strict accuracy    : {summary['strict_accuracy']:.3%}")
print(f"  mean overall score : {summary['mean_overall_score']:.3f}")
print(f"  hallucinations     : {summary['hallucinations']} "
      f"({summary['hallucination_rate']:.3%})")

# Per-model summary table
per_model: dict[str, dict] = {}
for model_name in MODEL_NAMES:
    model_rows = [r for r in all_rows if r["model_name"] == model_name]
    if not model_rows:
        continue
    ms = aggregate_json_results(model_rows, group_keys=("split", "task_type"))
    per_model[model_name] = {
        "n":               ms["n"],
        "strict_acc":      ms["strict_accuracy"],
        "mean_score":      ms["mean_overall_score"],
        "halluc_rate":     ms["hallucination_rate"],
        "elapsed_s":       round(model_elapsed.get(model_name, 0.0), 1),
    }

print("\n=== Per-model summary ===")
df = pd.DataFrame.from_dict(per_model, orient="index")
df.index.name = "model"
df["strict_acc"]  = df["strict_acc"].map("{:.3%}".format)
df["mean_score"]  = df["mean_score"].map("{:.3f}".format)
df["halluc_rate"] = df["halluc_rate"].map("{:.3%}".format)
display(df)

print("\n=== Per model × split × task ===")
for g, gs in summary["groups"].items():
    print(
        f"  {g:70s}  n={gs['n']:>3d}  "
        f"strict={gs['strict_accuracy']:.3%}  "
        f"mean={gs['mean_overall_score']:.3f}  "
        f"halluc={gs['hallucination_rate']:.3%}"
    )


In [ ]:
# ── Cell 8: Save Predictions & Report ────────────────────────────────────────
pred_path   = OUTPUT_DIR / "predictions.jsonl"
report_path = OUTPUT_DIR / "report.json"

pred_path.write_text(
    "\n".join(json.dumps(r, ensure_ascii=False, default=str) for r in all_rows)
)

report_obj = {
    "models":     MODEL_NAMES,
    "excluded":   list(EXCLUDE),
    "dataset":    HF_DATASET_ID,
    "subset":     {
        "max_per_task": MAX_PER_TASK,
        "perception":   len(subset["perception"]),
        "reasoning":    len(subset["reasoning"]),
    },
    "seed":            SEED,
    "per_model_elapsed_s": model_elapsed,
    "summary":     summary,
}
report_path.write_text(json.dumps(report_obj, indent=2, ensure_ascii=False, default=str))

print(f"Predictions  -> {pred_path}")
print(f"Report       -> {report_path}")
print(f"Total rows   : {len(all_rows)}")
